In [ ]:
# ----------------------------- Setup ----------------------------- #
%matplotlib tk
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.widgets import Button

prec = 100
RHP = RealField(prec)
CHP = ComplexField(prec)

# ----------------------------- Input ----------------------------- #
coeff_input = input(
    "Enter polynomial coefficients separated by spaces\n"
    "(highest degree first, e.g. 1 0 -4 ... 1):\n"
).strip()

coeffs_hp = [RHP(c) for c in coeff_input.split() if c]

if not coeffs_hp:
    raise ValueError("No coefficients entered!")

# ----------------------------- Polynomial निर्माण ----------------------------- #
R.<x> = RHP[]
f = sum(c * x^(len(coeffs_hp) - i - 1) for i, c in enumerate(coeffs_hp))
fC = f.change_ring(CHP)

# ----------------------------- Clean Printing ----------------------------- #
def format_coeff(c, digits=12, tol=1e-30):
    """Mathematica-like coefficient formatting"""
    if abs(c) < tol:
        return None  # drop tiny noise

    c = c.n(digits=digits)

    # Integer-like?
    if c.is_integer():
        return str(Integer(c))

    # Scientific notation if needed
    s = f"{c:.12g}"

    # Clean exponent formatting (Mathematica-like)
    if "e" in s:
        base, exp = s.split("e")
        exp = int(exp)
        return f"{base}*10^{exp}"

    return s

def polynomial_to_string(coeffs, var="x", digits=12):
    terms = []
    n = len(coeffs)

    for i, c in enumerate(coeffs):
        power = n - i - 1
        fc = format_coeff(c, digits)

        if fc is None:
            continue

        # Sign handling
        sign = "-" if fc.startswith("-") else "+"
        fc_abs = fc[1:] if fc.startswith("-") else fc

        # Build term
        if power == 0:
            term = f"{fc_abs}"
        elif power == 1:
            if fc_abs == "1":
                term = var
            else:
                term = f"{fc_abs}*{var}"
        else:
            if fc_abs == "1":
                term = f"{var}^{power}"
            else:
                term = f"{fc_abs}*{var}^{power}"

        terms.append((sign, term))

    if not terms:
        return "0"

    # First term keeps its sign if negative, drops '+' if positive
    first_sign, first_term = terms[0]
    result = (first_term if first_sign == "+" else f"-{first_term}")

    for sign, term in terms[1:]:
        result += f" {sign} {term}"

    return result

# Print polynomial
poly_str = polynomial_to_string(coeffs_hp)
print(f"\nPolynomial: f(x) = {poly_str} = 0")

# ----------------------------- Roots ----------------------------- #
roots = fC.roots(multiplicities=True)

# ----------------------------- Helpers ----------------------------- #
def poly_eval(coeffs, r):
    p = CHP(0)
    for c in coeffs:
        p = p * r + c
    return p

def relative_residual(coeffs, r):
    numerator = abs(poly_eval(coeffs, r))
    denom = sum(abs(c) * abs(r)^(len(coeffs) - i - 1) for i, c in enumerate(coeffs))
    return numerator / denom if denom != 0 else numerator

# ----------------------------- Print Roots ----------------------------- #
print(f"\nPolynomial degree: {len(coeffs_hp)-1}")
print("Roots with high precision, multiplicity and relative residuals:")

for i, (r, mult) in enumerate(roots, 1):
    res = relative_residual(coeffs_hp, r)
    print(f"Root {i}: {r.n(prec)} (multiplicity {mult})   Residual: {res:.2e}")

# ----------------------------- Plotting ----------------------------- #
roots_np = np.array([complex(r) for r, _ in roots])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

# ---- Complex Plane ----
ax1.axhline(0, color='gray', lw=1)
ax1.axvline(0, color='gray', lw=1)
ax1.scatter(roots_np.real, roots_np.imag, color='red', s=30)
ax1.set_title("Roots in Complex Plane")
ax1.set_xlabel("Re")
ax1.set_ylabel("Im")
ax1.grid(True)
ax1.set_aspect('auto')

# ---- Real Line Plot ----
real_roots = [r for r, mult in roots if abs(r.imag()) < RHP('1e-12')]

x_vals = np.linspace(float(min(roots_np.real)) - 1,
                     float(max(roots_np.real)) + 1, 2000)

y_vals = np.array([float(f(RHP(xx))) for xx in x_vals])

line, = ax2.plot(x_vals, y_vals, lw=1, label="f(x)")
ax2.axhline(0, color='gray', lw=1)

ax2.scatter(
    [float(r.real()) for r in real_roots],
    [float(fC(r).real()) for r in real_roots],
    color='red',
    s=60,
    zorder=5
)

ax2.set_title("Polynomial on Real Line")
ax2.set_xlabel("x")
ax2.set_ylabel("f(x)")
ax2.grid(True)
ax2.legend()

# ----------------------------- Buttons ----------------------------- #
max_abs = np.max(np.abs(y_vals))
linthresh = 1.0

def set_linear(event):
    ax2.set_yscale('linear')
    ax2.set_ylim(-max_abs * 1.05, max_abs * 1.05)
    fig.canvas.draw_idle()

def set_symlog(event):
    ax2.set_yscale('symlog', linthresh=linthresh, linscale=1.0)
    fig.canvas.draw_idle()

axlinear = plt.axes([0.8, 0.02, 0.1, 0.04])
axsymlog = plt.axes([0.65, 0.02, 0.1, 0.04])

b_linear = Button(axlinear, 'Linear')
b_symlog = Button(axsymlog, 'Symlog')

b_linear.on_clicked(set_linear)
b_symlog.on_clicked(set_symlog)

plt.tight_layout()
plt.show()